The aim of this project is to make an ice topography map with as high resolution as possible. By looking at each beam separately, we go from a footprint on the order of hundreds of meters to less than a hundred. 

The method is as follows:

1. Detect the ice-ocean (or ocean-atmosphere) interface from ADCP echo intensity for each ping and beam.
2. Derive unit vetors pointing in the beam directions of the ADCP in a earth referenced coordinate system for each beam and timestamp.
3. Use the AUV position, detected interface and the unit vectors to get point clouds with ice draft in the earth-referenced coordinate system.

Note that both the beam direction matrix and the intrument-to-earth rotation matrix depend on the ADCP used. 

In [ ]:
make_plots = True

import numpy as np
from pandas import DataFrame
from xarray import open_dataset, concat, broadcast
from pymap3d import geodetic2enu, enu2geodetic

from pinnacle45 import beam_direction_instrument__Pin45, roation_matrix_instrument2earth__Pin45
from adcp import distance_to_interface
from soundspeed import sound_speed_in_DIS_cavity, refine_vertical_distance
from density import z_from_p

if make_plots:
    import matplotlib.pyplot as plt
    from cartopy.crs import Mercator, PlateCarree
    from cmocean.cm import deep, ice
    from plot import nice_lonlat_gridlines

In [ ]:
hist_fig_size = (10,3)

### Load data

In [ ]:
mission = 'NBP2202_03'

In [ ]:
if mission == 'NBP2202_02':
    echo_intensity_file = 'data/derived/NBP2202_02_cleaned.nc'
    interpolate_range = True
elif mission == 'NBP2202_03':
    echo_intensity_file = 'data/derived/NBP2202_03_cleaned.nc'
    interpolate_range = True
elif mission == 'NBP2202_04':
    echo_intensity_file = 'data/derived/NBP2202_04_cleaned.nc'
    interpolate_range = True
if mission == 'NBP2202_02_no_interp':
    echo_intensity_file = 'data/derived/NBP2202_02_cleaned.nc'
    interpolate_range = False
elif mission == 'NBP2202_03_no_interp':
    echo_intensity_file = 'data/derived/NBP2202_03_cleaned.nc'
    interpolate_range = False
elif mission == 'NBP2202_04_no_interp':
    echo_intensity_file = 'data/derived/NBP2202_04_cleaned.nc'
    interpolate_range = False
ds = open_dataset(echo_intensity_file)
ds

### Detect ice-ocean/ocean-atmosphere interface from echo intensity data

In [ ]:
ds['distance_to_interface'] = distance_to_interface(ds, 
                                                    interpolate=interpolate_range, 
                                                    rmin=200, 
                                                    rmax=1150,  
                                                    ampmin=100,
                                                   )   
if make_plots:
    ds.distance_to_interface.plot.hist(bins=100, figsize = hist_fig_size);

### Derive unit vectors along beam directions

In [ ]:
beam_direction__inst = beam_direction_instrument__Pin45()
inst2earth = roation_matrix_instrument2earth__Pin45(ds,
                                                    pitch='pitch_adcp', 
                                                    roll='roll_adcp', 
                                                    heading='heading_adcp', 
                                                    time_dim='time')
ds['beam_direction__earth'] = inst2earth @ beam_direction__inst 

### AUV position in cartesian coordinate system

We need to have the AUV position in a cartestian coordinate system (eastings and northings).

In [ ]:
lon   = ds['longitude']
lat   = ds['latitude']
z     = z_from_p(ds['pressure'])

# Longitude and latitude of origo
lon0 = lon.mean()
lat0 = lat.mean()

(e,n) = geodetic2enu(lat, lon, 0, lat0, lon0, 0)[0:2]
ds['AUV_position'] = concat([e,n,z], dim='earth')

In [ ]:
def plot_in_enu(da, **kwargs):
    plt.scatter(da.sel(earth='E'), da.sel(earth='N'), 
                c=-da.sel(earth='U'), s=5, cmap=deep, **kwargs)
    plt.xlabel('Eastings (m)')
    plt.ylabel('Northings (m)')
    plt.gca().axis('equal')
    cbar = plt.colorbar(label='depth (m)')
    cbar.ax.invert_yaxis()  

In [ ]:
if make_plots:
    plot_in_enu(ds.AUV_position)
    plt.title('AUV position')

## Derive ice draft

In [ ]:
ds['ice_draft_raw'] = ds.AUV_position + (ds.distance_to_interface * ds.beam_direction__earth)

### Refine ice draft using sound speed profile
Improve estimate of vertical distance to interface using a depth-dependent model for speed of sound and ray theory. Let $t_0$ be the measured time between sensor and detected interface, i.e.
$$
t_0 = \frac{d}{c_\text{adcp}}
$$
where $d$ is the along-beam distance and $c_\text{adcp}$ is the sound speed used by the ADCP. We're going to estimate the vertical distance between ADCP and detected interface using the following iteration scheme:

Let $h_0$ be the vertical distance assuming stratight path, depth averaged sound speed $c_\text{av}$ and beam angle $\theta_0$ (at sensor depth $z_0$):
$$
h_0 = c_\text{adcp} t_0 \sin\theta_0 = d\sin\theta_0
$$

Iteration scheme:

1. Compute travel time between $z_0$ (sensor depth) and $z0 + h_i$ (estimated ice draft) using eq 8 in Hovem 2013:
$$
    \tau_i = \int_{z_0}^{z_0 + h_i} \frac{dz}{c(z)\sqrt{1-\left(\xi c(z)\right)^2}}
$$
2. Update vertical distance:
$$
    h_{i+1} = h_i + (t_0 - \tau_i) c_\text{av}
$$

In [ ]:
sound_speed_profile = sound_speed_in_DIS_cavity(make_plot=make_plots)

In [ ]:
theta0 = np.degrees(np.arcsin(ds.beam_direction__earth.sel(earth='U')))   # vertical beam angle (90 deg = along z-axis )
t0     = ds.distance_to_interface/ds.sound_speed                          # measured travel time between sensor and detected interface
h0     = ds.ice_draft_raw.sel(earth='U') - ds.AUV_position.sel(earth='U') # estimated vertical distance

if make_plots:
    theta0.plot.hist(bins=100, figsize =  hist_fig_size);
    plt.title('Vertical beam angle (90 deg = along z-axis )')
    plt.xlabel('theta (deg)')
    plt.show()

    t0.plot.hist(bins=100, figsize =  hist_fig_size);
    plt.title('Measured travel time between sensor and detected interface')
    plt.xlabel('travel time (s)')
    plt.show()

    h0.plot.hist(bins=100, figsize =  hist_fig_size);
    plt.title('First estimate of vertical distance between sensor and detected interface')
    plt.xlabel('h0 (m)')
    plt.show()

In [ ]:
ds['ice_draft'] = ds['ice_draft_raw'].copy()
h = np.zeros([ds.sizes['time'], ds.sizes['beam']])

for t in range(ds.sizes['time']):
    for b in range(ds.sizes['beam']):
        h[t,b] = refine_vertical_distance(t0.isel(time=t, beam=b).values, 
                                          -ds.AUV_position.sel(earth='U').isel(time=t).values, 
                                          h0.isel(time=t, beam=b).values, 
                                          theta0.isel(time=t, beam=b).values, 
                                          sound_speed_profile, 
                                          N=3, make_plots=False, fast_integration=True)

ds['refined_vertical_distance'] = (('time', 'beam'), h)    
ds['ice_draft'][2,:,:] = ds.AUV_position.sel(earth='U') + ds['refined_vertical_distance']

if make_plots:
    ds['refined_vertical_distance'].plot.hist(bins=100, figsize=hist_fig_size)
    plt.show()

    (ds.ice_draft_raw - ds.ice_draft).sel(earth='U').plot.hist(bins=100, figsize=hist_fig_size);
    plt.title('Ice draft adjustment due to sound speed corrections')
    plt.xlabel('Adjustment (m)')

In [ ]:
if make_plots:
    fig, axes = plt.subplots(2,2,figsize=(10,10), layout='tight')
    for (b,ax) in zip (ds.beam.values, axes.ravel()):
        plt.sca(ax)
        plot_in_enu(ds.ice_draft.sel(beam=b))
        plt.title(f'Ice draft beam {b}')

### Convert back to longitude/latitude

In [ ]:
(la,lo) = enu2geodetic(ds.ice_draft.sel(earth='E'), 
                       ds.ice_draft.sel(earth='N'), 
                       ds.ice_draft.sel(earth='U'), 
                       lat0, lon0, 0)[0:2]
ds['ice_draft_depth'] = -ds.ice_draft.sel(earth='U')
ds['ice_draft_latitude'] = (('time', 'beam'), la)
ds['ice_draft_longitude'] = (('time', 'beam'), lo)

### Save as csv

In [ ]:
def make_1d(original, shape_as):
    """
    Broadcast to desired dimensions and convert to 1d array
    """
    return broadcast(original, shape_as)[0].stack(z=shape_as.dims)

In [ ]:
data = {'latitude'    : make_1d(ds.ice_draft_latitude, ds.ice_draft_depth),
        'longitude'   : make_1d(ds.ice_draft_longitude, ds.ice_draft_depth),
        'ice_draft'   : make_1d(ds.ice_draft_depth, ds.ice_draft_depth),
        'beam'        : make_1d(ds.beam, ds.ice_draft_depth),
        'timestamp'   : make_1d(ds.time_str, ds.ice_draft_depth),
        'AUV_depth'   : make_1d(-ds.AUV_position.sel(earth='U'), ds.ice_draft_depth),
        'AUV_lat'     : make_1d(ds.latitude, ds.ice_draft_depth),
        'AUV_lon'     : make_1d(ds.longitude, ds.ice_draft_depth),
        'ADCP_roll'   : make_1d(ds.roll_adcp, ds.ice_draft_depth),
        'ADCP_pitch'  : make_1d(ds.pitch_adcp, ds.ice_draft_depth),
        'ADCP_heading': make_1d(ds.heading_adcp, ds.ice_draft_depth)}

ice_draft_data = DataFrame(data)
ice_draft_data = ice_draft_data.dropna(axis='index', how='any')
ice_draft_data

In [ ]:
if make_plots:
    plt.figure(figsize=(6,6), layout='tight')
    ax = plt.axes(projection = Mercator(central_longitude = lon0.values,
                                        latitude_true_scale = lat0.values))
    
    im = ax.scatter(ice_draft_data['longitude'], ice_draft_data['latitude'],
                    c = ice_draft_data['ice_draft'], s=5, transform=PlateCarree(), cmap = ice, 
                    vmin=np.nanpercentile(ice_draft_data['ice_draft'],5), 
                    vmax=np.nanpercentile(ice_draft_data['ice_draft'],95)) 
    nice_lonlat_gridlines(ax, alpha=0.3)
    cbar = plt.colorbar(im, extend='both', shrink=0.7, label='Ice draft (m)')
    cbar.ax.invert_yaxis()

In [ ]:
savefile = f'data/derived/{mission}_ice_draft.csv'
ice_draft_data.to_csv(savefile, index=False)
print(f'Saved ice draft as: {savefile}')